In [ ]:
import pandas as pd
import requests
import json
import zipfile

# Get Institutions
with zipfile.ZipFile("../data/input.zip") as z:
    with z.open("input/fdic/cert_nums.csv") as f:
        banks_id = pd.read_csv(f)

import requests

def get_bank_data(certs):
    url = "https://banks.data.fdic.gov/api/financials"

    cert_filter = " OR ".join(f"CERT:{c}" for c in certs)
    repdte_filter = "REPDTE:20211231"

    filter_expr = f"({cert_filter}) AND {repdte_filter}"
    
    params = dict(
        filters  = filter_expr,
        fields     = "CERT,REPDTE,ASSET,DEP,DEPINS",
        sort_by    = "REPDTE",
        sort_order = "DESC",
        limit      = 10000,
        offset     = 0,
        agg_term_fields = "REPDTE",
        agg_sum_fields  = "ASSET,DEP,DEPINS",
        agg_limit       = 1
    )
    
    resp = requests.get(url, params=params)
    resp.raise_for_status()

    data = resp.json()["data"]
    return data

output = [get_bank_data(certs)]

bank_data = (
    pd.DataFrame(
        d["data"]
        for records in output
        for d in records)
    .rename(columns={"ASSET": "ASSETS"})
    .sort_values(["CERT", "REPDTE"])
    .drop_duplicates("CERT", keep="last")
    [["CERT", "DEP", "DEPINS", "ASSETS"]]
    .reset_index(drop=True)
)

bank_data_final = (bank_data
    .merge(banks_id, 
           left_on = "CERT", right_on = "fdic_cert_num",
           how = 'right')
    .sort_values(["CERT", "ID_RSSD_PARENT"])
)

# bank_data_final.to_csv("data/input/fdic/bank_deposits.csv")


In [ ]:
bank_data_final[bank_data_final['CERT'].isna()]

In [ ]:
def get_bank_data(bank):
    cert_num = int(bank)
    # print(bank)
    params = dict(
        filters    = "CERT:%d" % cert_num,
        fields     = "CERT,REPDTE,ASSET,DEP,DEPINS",
        sort_by    = "REPDTE",
        sort_order = "DESC",
        limit      = 10000,
        offset     = 0,
        agg_term_fields = "REPDTE",
        agg_sum_fields  = "ASSET,DEP,DEPINS",
        agg_limit       = 1
    )
    resp = requests.get(url=url, params = params)
    data = resp.json()
    if data["data"]:
        df = {x: data["data"][0]["data"][x] 
                          for x in ["CERT", "DEP", "DEPINS", "ASSET"]}
        return df

In [ ]:
import pandas as pd
import requests
import json
import zipfile

# Get Institutions
with zipfile.ZipFile("../data/input.zip") as z:
    with z.open("input/fdic/cert_nums.csv") as f:
        banks_id = pd.read_csv(f)

import requests

def get_bank_data(certs):
    url = "https://banks.data.fdic.gov/api/financials"

    cert_filter = " OR ".join(f"CERT:{c}" for c in certs)
    repdte_filter = "REPDTE:20250930"

    filter_expr = f"({cert_filter}) AND {repdte_filter}"
    
    params = dict(
        filters  = filter_expr,
        fields     = "CERT,REPDTE,ASSET,DEP,DEPINS",
        sort_by    = "REPDTE",
        sort_order = "DESC",
        limit      = 10000,
        offset     = 0,
        agg_term_fields = "REPDTE",
        agg_sum_fields  = "ASSET,DEP,DEPINS",
        agg_limit       = 1
    )
    
    resp = requests.get(url, params=params)
    resp.raise_for_status()

    data = resp.json()["data"]
    return data

def batched(iterable, n=30):
    iterable = list(iterable)
    for i in range(0, len(iterable), n):
        yield iterable[i:i+n]


output = [get_bank_data(cert_batch) for cert_batch in batched(certs)]

bank_data = (
    pd.DataFrame(
        d["data"]
        for records in output
        for d in records)
    .rename(columns={"ASSET": "ASSETS"})
    .sort_values(["CERT", "REPDTE"])
    .drop_duplicates("CERT", keep="last")
    [["CERT", "DEP", "DEPINS", "ASSETS"]]
    .reset_index(drop=True)
)

bank_data_final = (bank_data
    .merge(right=banks_id, 
           left_on = "CERT", right_on = "fdic_cert_num")
    .sort_values(["CERT", "ID_RSSD_PARENT"])
)

bank_data_final.to_csv("../data/input/fdic/bank_deposits.csv")

In [ ]:
import pandas as pd
import requests
import json
import zipfile

# Get Institutions
with zipfile.ZipFile("../data/input.zip") as z:
    with z.open("input/fdic/cert_nums.csv") as f:
        banks_id = pd.read_csv(f)

import requests

def get_bank_data(certs):
    url = "https://banks.data.fdic.gov/api/financials"

    cert_filter = " OR ".join(f"CERT:{c}" for c in certs)
    repdte_filter = "REPDTE:20211231"

    filter_expr = f"({cert_filter}) AND {repdte_filter}"
    
    params = dict(
        filters  = filter_expr,
        fields     = "CERT,REPDTE,ASSET,DEP,DEPINS",
        sort_by    = "REPDTE",
        sort_order = "DESC",
        limit      = 10000,
        offset     = 0,
        agg_term_fields = "REPDTE",
        agg_sum_fields  = "ASSET,DEP,DEPINS",
        agg_limit       = 1
    )
    
    resp = requests.get(url, params=params)
    resp.raise_for_status()

    data = resp.json()["data"]
    return data


certs = banks_id["fdic_cert_num"].to_list()

bank_data = (
    pd.DataFrame(d['data'] for d in get_bank_data(certs))
    .rename(columns={"ASSET": "ASSETS"})
    .sort_values(["CERT", "REPDTE"])
    .drop_duplicates("CERT", keep="last")
    [["CERT", "DEP", "DEPINS", "ASSETS"]]
    .reset_index(drop=True)
)

bank_data_final = (bank_data
    .merge(right=banks_id, 
           left_on = "CERT", right_on = "fdic_cert_num")
    .sort_values(["CERT", "ID_RSSD_PARENT"])
)

bank_data_final.to_csv("../data/input/fdic/bank_deposits.csv")

In [ ]:
pd.DataFrame(d['data'] for d in get_bank_data([27591]))

In [ ]:
def get_bank_data(certs):
    url = "https://banks.data.fdic.gov/api/financials"

    cert_filter = " OR ".join(f"CERT:{c}" for c in certs)
    repdte_filter = "REPDTE:20211231"

    filter_expr = f"({cert_filter}) AND {repdte_filter}"
    # filter_expr = f"{cert_filter}"
    
    params = dict(
        filters  = filter_expr,
        fields     = "CERT,REPDTE,ASSET,DEP,DEPINS",
        sort_by    = "REPDTE",
        sort_order = "DESC",
        limit      = 10000,
        offset     = 0,
        agg_term_fields = "REPDTE",
        agg_sum_fields  = "ASSET,DEP,DEPINS",
        agg_limit       = 1
    )
    
    resp = requests.get(url, params=params)
    resp.raise_for_status()

    data = resp.json()["data"]
    return data

In [ ]:
get_bank_data([14])

In [ ]:
# df = [pd.DataFrame(d["data"]) for d in i for i in output]

len(output)


In [ ]:
import pandas as pd



In [ ]:
df